In [1]:
from ta.volatility import AverageTrueRange

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from backtesting import Strategy
from backtesting import Backtest


from datetime import timedelta

import sys
sys.path.append('../utils')
import Util
import importlib
importlib.reload(Util)

from enum import Enum
import time

from random import randint
import random

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
pair = 'GBPJPY'

df_d1_raw = pd.read_csv(f"../Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h4_raw = pd.read_csv(f"../Trading Data/{pair}_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_m15_raw = pd.read_csv(f"../Trading Data/{pair}_M15.csv",sep=',', parse_dates=['Date'], index_col='Date')

print(df_d1_raw)
print(df_h4_raw)
print(df_m15_raw)

               Open     High      Low    Close  Tickvol
Date                                                   
2010-09-10  129.605  130.065  128.995  129.255   172229
2010-09-13  129.565  130.115  128.780  129.090   215859
2010-09-14  129.105  129.430  127.640  129.015   226170
2010-09-15  129.015  134.070  128.555  133.975   252317
2010-09-16  133.975  134.260  132.970  134.150   222443
...             ...      ...      ...      ...      ...
2026-09-18  208.323  211.276  208.233  210.095   364707
2026-09-21  210.296  210.762  209.676  210.321   243731
2026-09-22  210.321  210.892  209.513  210.080   333200
2026-09-23  210.080  210.176  209.422  209.564   273936
2026-09-24  209.566  210.105  208.768  209.848   199988

[4183 rows x 5 columns]
                        Open     High      Low    Close  Tickvol
Date                                                            
2010-09-10 04:00:00  129.605  129.630  128.995  129.385    25539
2010-09-10 08:00:00  129.380  129.985  129.275  129.

In [3]:
structure_d1_window = 2
structure_m15_window = 2

date_format = '%Y-%m-%d %H:%M'
# start_date = '2023-1-1 00:00'
# end_date = '2024-3-29 00:00'
start_date = datetime.strptime('2021-1-1 00:00', date_format)
end_date = datetime.strptime('2026-9-16 00:00', date_format)
# start_date = '2022-01-1 00:00'

timezone = "Europe/London"

In [4]:
# process D1 data
df_d1 = df_d1_raw.loc[start_date - timedelta(days=30) : end_date]

df_d1 = Util.implement_market_structure_states_fixed_pivots(df_d1, n=structure_d1_window)
df_d1["state_changed"] = df_d1["market_state"] != df_d1[
    "market_state"
].shift(1)
print(df_d1[df_d1["state_changed"]][["Close", "market_state"]])
df_d1 = df_d1.loc[start_date : end_date]

              Close market_state
Date                            
2020-12-02  139.632          NaN
2020-12-03  139.680          NaN
2020-12-04  139.937          NaN
2020-12-07  139.114          NaN
2020-12-08  139.134          NaN
...             ...          ...
2026-04-30  213.107      BEARISH
2026-05-25  214.592      BULLISH
2026-06-18  212.961      BEARISH
2026-06-29  214.706      BULLISH
2026-07-30  215.005      BEARISH

[82 rows x 2 columns]


In [5]:
# process h4 data
df_h4 = df_h4_raw.loc[start_date - timedelta(days=20) : end_date]
df_h4 = Util.implement_market_structure_states_fixed_pivots(df_h4, 5)
df_h4 = df_h4.loc[start_date : end_date]

h4_d1_direction_list = []
for i in range(len(df_d1)):
    row = df_d1.iloc[i]
    start_date_daily = row.name
    end_date_daily = start_date_daily + timedelta(hours=23, minutes=59)
    df_by_day = df_h4.loc[f'{start_date_daily}' : f'{end_date_daily}']

    for index2, row2 in df_by_day.iterrows():
        h4_d1_direction_list.append(df_d1.iloc[i-1]['market_state'])

df_h4['market_state_d1'] = h4_d1_direction_list

indicator_atr_14 = AverageTrueRange(df_h4['High'], df_h4['Low'], df_h4['Close'], window=14)
df_h4['ATR'] = indicator_atr_14._atr

df_h4

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,market_state_d1,ATR
Date,,,,,,,,,,
2021-01-04 00:00:00,140.944,141.304,140.776,141.005,37352,True,False,BULLISH,BEARISH,0.000000
2021-01-04 04:00:00,141.007,141.043,140.847,140.938,20550,False,False,BULLISH,BEARISH,0.000000
2021-01-04 08:00:00,140.936,141.064,140.412,140.757,47343,False,False,BULLISH,BEARISH,0.000000
2021-01-04 12:00:00,140.758,140.861,140.188,140.226,55445,False,False,BULLISH,BEARISH,0.000000
2021-01-04 16:00:00,140.227,140.233,139.687,139.870,55631,False,False,BULLISH,BEARISH,0.000000
...,...,...,...,...,...,...,...,...,...,...
2026-09-15 08:00:00,208.652,209.090,208.514,208.655,56407,False,False,BEARISH,BEARISH,0.567701
2026-09-15 12:00:00,208.655,209.211,208.607,209.168,74480,False,False,BULLISH,BEARISH,0.570294
2026-09-15 16:00:00,209.167,209.224,208.885,209.066,38410,False,False,BULLISH,BEARISH,0.553773


In [6]:
# process m15 data

df_m15 = df_m15_raw.loc[start_date - timedelta(hours=3): end_date]
df_m15 = Util.implement_market_structure_states_fixed_pivots(df_m15, n=structure_m15_window)

m15_daily_direction_list = []

index_list = []

indicator_atr_m15 = AverageTrueRange(df_m15['High'], df_m15['Low'], df_m15['Close'], window=14)
df_m15['ATR'] = indicator_atr_m15._atr

df_m15 = df_m15.loc[start_date : end_date]
for i in range(len(df_h4)):
    row = df_h4.iloc[i]
    start_date_h4 = row.name
    end_date_h4 = start_date_h4 + timedelta(hours=3, minutes=59)
    df_by_day = df_m15.loc[f'{start_date_h4}' : f'{end_date_h4}']

    for index2, row2 in df_by_day.iterrows():
        m15_daily_direction_list.append(df_h4.iloc[i]['market_state_d1'])

        # ajust to daylight saving time
        dst = pd.Timestamp(index2, tz=timezone).dst()
        if dst == timedelta(0):
            index2 = index2 - timedelta(hours=1)
            index_list.append(index2)
        else:
            index_list.append(index2)

df_m15['market_state_d1'] = m15_daily_direction_list

df_m15.index = index_list

In [ ]:
# add some sl tp management

class RandomBiasStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    trades_list = None
    stoploss_window = 150

    def is_signal(self, date):
        for i in range(0, len(self.trades_list)):
            if self.trades_list[i] == date:
                return True
        return False

    def get_highest_high(self):
        return self.data.High[len(self.data.High) - self.stoploss_window : len(self.data.High)].max()
    
    def get_lowest_low(self):
        return self.data.Low[len(self.data.Low) - self.stoploss_window : len(self.data.Low)].min()
    
    def init(self):
        super().init()

    def next(self):
        super().next()
        if self.is_signal(self.data.index[-1]) and len(self.trades) == 0 and len(self.closed_trades) < 63:
            if self.data['market_state_d1'][-1] == 'BEARISH':
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_lowest_low())
                if sl == 0:
                    sl = round(random.uniform(0.3,1), 1)
                tp = self.TPcoef * sl

                sl1 = planned_entry_price - sl
                tp1 = planned_entry_price + tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.buy(sl=sl1, tp=tp1, size=position_size)
            else:
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_highest_high())
                if sl == 0:
                    sl = round(random.uniform(0.3,1), 1)
                tp = self.TPcoef * sl

                sl1 = planned_entry_price + sl
                tp1 = planned_entry_price - tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.sell(sl=sl1, tp=tp1, size=position_size)

In [ ]:
trades_list = []
while len(trades_list) != 73:
    random_index = randint(0, len(df_m15))
    candle = df_m15.iloc[random_index]
    # exclu Saturday and Sunday
    if candle.name.weekday() < 5:
        if 9 <= candle.name.hour and candle.name.hour <= 19:
            trades_list.append(candle.name)
trades_list.sort()

bt = Backtest(df_m15, RandomBiasStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run(trades_list=trades_list)

print(stats)

bt.plot()

Start                     2021-01-03 23:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 01:00:00
Exposure Time [%]                    15.82804
Equity Final [$]                      9455.05
Equity Peak [$]                     10058.825
Return [%]                            -5.4495
Buy & Hold Return [%]                48.17505
Return (Ann.) [%]                    -0.87596
Volatility (Ann.) [%]                 4.03178
CAGR [%]                             -0.97869
Sharpe Ratio                         -0.21726
Sortino Ratio                        -0.31596
Calmar Ratio                         -0.10401
Alpha [%]                             -7.1519
Beta                                  0.03534
Max. Drawdown [%]                    -8.42151
Avg. Drawdown [%]                    -3.16061
Max. Drawdown Duration     1040 days 08:30:00
Avg. Drawdown Duration      403 days 06:45:00
# Trades                                   62
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p1261', ...)

In [10]:
i = 0
expectancy_list = []
winrate_list = []

while len(expectancy_list) < 1000:
    trades_list = []
    while len(trades_list) != 80:
        random_index = randint(0, len(df_m15)-1)
        candle = df_m15.iloc[random_index -1]
        # exclu Saturday and Sunday
        if candle.name.weekday() < 5:
            if 9 <= candle.name.hour and candle.name.hour <= 19:
                trades_list.append(candle.name)
    trades_list.sort()

    bt = Backtest(df_m15, RandomBiasStrat, cash=10000, margin=0.009, finalize_trades=True) #0.0002
    stats = bt.run(trades_list=trades_list)
    if stats['# Trades'] == 63:
        expectancy_list.append(stats['Expectancy [%]'])
        winrate_list.append(stats['Win Rate [%]'])
        print(f'{i} {len(expectancy_list)} {stats['Expectancy [%]']} Done')
    i += 1

print(expectancy_list)

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=81980: Broker canceled the order due to insufficient margin (equity=10886.48, margin_available=10886.48).
  warnings.warn(


0 1 0.2691696250268581 Done
1 2 -0.0611809980898309 Done
3 3 0.0998750572177188 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=129788: Broker canceled the order due to insufficient margin (equity=9588.27, margin_available=9588.27).
  warnings.warn(


4 4 0.04778005389247523 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=53101: Broker canceled the order due to insufficient margin (equity=9679.43, margin_available=9679.43).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=8793: Broker canceled the order due to insufficient margin (equity=10196.29, margin_available=10196.29).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=139500: Broker canceled the order due to insufficient margin (equity=11644.44, margin_available=11644.44).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=28598: Broker canceled the order due to insufficient margin (equity=10176.47, margin_available=10176.47).
  warni

...

1332 994 0.3000022175206462 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=113807: Broker canceled the order due to insufficient margin (equity=10252.27, margin_available=10252.27).
  warnings.warn(


1334 995 0.09636158092956303 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=135447: Broker canceled the order due to insufficient margin (equity=11412.91, margin_available=11412.91).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=120141: Broker canceled the order due to insufficient margin (equity=13145.90, margin_available=13145.90).
  warnings.warn(


1336 996 0.31676496635678036 Done
1337 997 -0.1223081168443362 Done
1338 998 0.12496230282427506 Done
1339 999 0.10632175501635302 Done
1340 1000 0.25615782563644657 Done
[np.float64(0.2691696250268581), np.float64(-0.0611809980898309), np.float64(0.0998750572177188), np.float64(0.04778005389247523), np.float64(0.5042222022964119), np.float64(0.2724043694358593), np.float64(0.02637710237863256), np.float64(0.17784780556046922), np.float64(0.014564717235624134), np.float64(0.10124523159701694), np.float64(0.07743534188015042), np.float64(0.222209973058928), np.float64(0.023125860709268087), np.float64(-0.14584527688365603), np.float64(0.1413400877464019), np.float64(0.04911173030962503), np.float64(0.29055230709364965), np.float64(0.061537322279674414), np.float64(-0.005705965732346942), np.float64(-0.0027308410936182047), np.float64(0.10911785112979629), np.float64(0.10668840067438654), np.float64(0.14742818098349122), np.float64(0.1455996332873021), np.float64(0.108261916085177), np.f

In [12]:
print(len(expectancy_list))

p95 = np.percentile(expectancy_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(expectancy_list)}")
mean_expectancy = np.mean(expectancy_list)
print(f"mean = {mean_expectancy}")
print(f"min = {min(expectancy_list)}")
print(f"gap = {0.34589 - mean_expectancy}")

df_result = pd.DataFrame({'expectancy' : expectancy_list})
rank = (df_result['expectancy'] <= 0.34589).mean() * 100 
print(f"rank = {rank}")
p_value = ((df_result['expectancy'] >= 0.34589).sum() + 1) / (len(df_result['expectancy']) + 1)
print(f"p_value = {p_value}")

print(len(winrate_list))

p95_winrate = np.percentile(winrate_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95_winrate}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(winrate_list)}")
mean_winrate = np.mean(winrate_list)
print(f"mean = {mean_winrate}")
print(f"min = {min(winrate_list)}")
print(f"gap = {45.09804 - mean_winrate}")

1000
p95 = 0.3179945297674906
max = 0.5150300042625569
mean = 0.07396891569310413
min = -0.36260296423928756
gap = 0.2719210843068959
rank = 97.0
p_value = 0.030969030969030968
1000
p95 = 47.61904761904761
max = 53.96825396825397
mean = 36.55238095238094
min = 19.047619047619047
gap = 8.545659047619054


In [14]:
# add some sl tp management

class RandomStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    trades_list = None
    stoploss_window = 150

    def is_signal(self, date):
        for i in range(0, len(self.trades_list)):
            if self.trades_list[i] == date:
                return True
        return False

    def get_highest_high(self):
        return self.data.High[len(self.data.High) - self.stoploss_window : len(self.data.High)].max()
    
    def get_lowest_low(self):
        return self.data.Low[len(self.data.Low) - self.stoploss_window : len(self.data.Low)].min()
    
    def init(self):
        super().init()

    def next(self):
        super().next()
        if self.is_signal(self.data.index[-1]) and len(self.trades) == 0 and len(self.closed_trades) < 63:
            direction = randint(1,2)
            if direction == 1:
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_lowest_low())
                if sl == 0:
                    sl = round(random.uniform(0.3,1), 1)
                tp = self.TPcoef * sl

                sl1 = planned_entry_price - sl
                tp1 = planned_entry_price + tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.buy(sl=sl1, tp=tp1, size=position_size)
            else:
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_highest_high())
                if sl == 0:
                    sl = round(random.uniform(0.3,1), 1)
                tp = self.TPcoef * sl

                sl1 = planned_entry_price + sl
                tp1 = planned_entry_price - tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.sell(sl=sl1, tp=tp1, size=position_size)

In [17]:
trades_list = []
while len(trades_list) != 73:
    random_index = randint(0, len(df_m15))
    candle = df_m15.iloc[random_index]
    # exclu Saturday and Sunday
    if candle.name.weekday() < 5:
        if 9 <= candle.name.hour and candle.name.hour <= 19:
            trades_list.append(candle.name)
trades_list.sort()

bt = Backtest(df_m15, RandomStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run(trades_list=trades_list)

print(stats)

bt.plot()

Start                     2021-01-03 23:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 01:00:00
Exposure Time [%]                    17.23941
Equity Final [$]                    10506.452
Equity Peak [$]                     12211.104
Return [%]                            5.06452
Buy & Hold Return [%]                48.17505
Return (Ann.) [%]                     0.77871
Volatility (Ann.) [%]                 4.12986
CAGR [%]                              0.87088
Sharpe Ratio                          0.18856
Sortino Ratio                         0.28782
Calmar Ratio                          0.05578
Alpha [%]                              3.7309
Beta                                  0.02768
Max. Drawdown [%]                   -13.95985
Avg. Drawdown [%]                    -0.45245
Max. Drawdown Duration      677 days 20:45:00
Avg. Drawdown Duration       10 days 20:11:00
# Trades                                   63
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p2113', ...)

In [18]:
i = 0
expectancy_list = []
winrate_list = []

while len(expectancy_list) < 1000:
    trades_list = []
    while len(trades_list) != 80:
        random_index = randint(0, len(df_m15)-1)
        candle = df_m15.iloc[random_index -1]
        # exclu Saturday and Sunday
        if candle.name.weekday() < 5:
            if 9 <= candle.name.hour and candle.name.hour <= 19:
                trades_list.append(candle.name)
    trades_list.sort()

    bt = Backtest(df_m15, RandomStrat, cash=10000, margin=0.009, finalize_trades=True) #0.0002
    stats = bt.run(trades_list=trades_list)
    if stats['# Trades'] == 63:
        expectancy_list.append(stats['Expectancy [%]'])
        winrate_list.append(stats['Win Rate [%]'])
        print(f'{i} {len(expectancy_list)} {stats['Expectancy [%]']} Done')
    i += 1

print(expectancy_list)

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=109770: Broker canceled the order due to insufficient margin (equity=8437.02, margin_available=8437.02).
  warnings.warn(


0 1 -0.27336741182763796 Done
1 2 -0.15138021305477284 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=1781: Broker canceled the order due to insufficient margin (equity=10199.94, margin_available=10199.94).
  warnings.warn(


2 3 0.100449815720802 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=112884: Broker canceled the order due to insufficient margin (equity=12867.79, margin_available=12867.79).
  warnings.warn(


...

1233 996 0.056821664318646325 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=67525: Broker canceled the order due to insufficient margin (equity=11333.14, margin_available=11333.14).
  warnings.warn(


1234 997 0.021261197983327193 Done
1236 998 -0.0208740864077115 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=102575: Broker canceled the order due to insufficient margin (equity=11301.18, margin_available=11301.18).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=126370: Broker canceled the order due to insufficient margin (equity=10354.67, margin_available=10354.67).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=54825: Broker canceled the order due to insufficient margin (equity=11024.00, margin_available=11024.00).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=74436: Broker canceled the order due to insufficient margin (equity=12159.49, margin_available=12159.49).
  w

1239 999 0.00537410696607635 Done
1241 1000 0.1289811912436708 Done
[np.float64(-0.27336741182763796), np.float64(-0.15138021305477284), np.float64(0.100449815720802), np.float64(0.3888232620134494), np.float64(-0.15359211253621202), np.float64(-0.038591110016937526), np.float64(-0.169087628581092), np.float64(0.025588269989622037), np.float64(-0.155896245266407), np.float64(-0.04553450836466216), np.float64(-0.007633737639281943), np.float64(0.021816109588562636), np.float64(0.2406842370483973), np.float64(-0.17385610919281258), np.float64(-0.14136100419268938), np.float64(0.11570116319072933), np.float64(-0.05542319751375856), np.float64(0.03564306855899518), np.float64(0.14724494347229342), np.float64(0.07990592812957024), np.float64(0.02259938907513159), np.float64(0.02583227801943822), np.float64(0.06936741175189737), np.float64(0.28134731607848673), np.float64(-0.12257671607457529), np.float64(0.08150464222920611), np.float64(0.006727818131435637), np.float64(0.11997793437066537)

In [19]:
print(len(expectancy_list))

p95 = np.percentile(expectancy_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(expectancy_list)}")
mean_expectancy = np.mean(expectancy_list)
print(f"mean = {mean_expectancy}")
print(f"min = {min(expectancy_list)}")
print(f"gap = {0.34589 - mean_expectancy}")

df_result = pd.DataFrame({'expectancy' : expectancy_list})
rank = (df_result['expectancy'] <= 0.34589).mean() * 100 
print(f"rank = {rank}")
p_value = ((df_result['expectancy'] >= 0.34589).sum() + 1) / (len(df_result['expectancy']) + 1)
print(f"p_value = {p_value}")

print(len(winrate_list))

p95_winrate = np.percentile(winrate_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95_winrate}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(winrate_list)}")
mean_winrate = np.mean(winrate_list)
print(f"mean = {mean_winrate}")
print(f"min = {min(winrate_list)}")
print(f"gap = {45.09804 - mean_winrate}")

1000
p95 = 0.2572644889683005
max = 0.41084528440672385
mean = 0.021436837101143504
min = -0.3552564562690981
gap = 0.32445316289885645
rank = 98.8
p_value = 0.012987012987012988
1000
p95 = 46.03174603174603
max = 52.38095238095239
mean = 35.34920634920635
min = 19.047619047619047
gap = 9.74883365079365
